# 4 Snn Benchmark

Train the LIF-SNN for each encoder, 5 folds x 3 seeds = 150 runs, then evaluate on the held-out test set.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports + Configuration                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import snntorch as snn
from snntorch import surrogate
import snntorch.functional as SF
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    average_precision_score, balanced_accuracy_score, matthews_corrcoef
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy
import time
import json
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
ENCODED_JUNE  = config.ENCODED_DIR
RESULTS_DIR   = config.RESULTS_DIR
CKPT_DIR      = os.path.join(RESULTS_DIR, 'checkpoints')
TENSORBOARD_DIR = os.path.join(RESULTS_DIR, 'tensorboard')

for d in [RESULTS_DIR, CKPT_DIR, TENSORBOARD_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Encoder names + colours (updated to final 10) ─────────────────────────────
ENCODER_NAMES = [
    'Rate', 'Phase', 'TTFS',
    'Delta-Mod', 'PDE', 'ATDE', 'MASTE',
    'ST-MW', 'AMSTE', 'SFBE',
]

ENCODER_COLORS = {
    'Rate':      '#2196F3',
    'Phase':     '#FF9800',
    'TTFS':      '#4CAF50',
    'Delta-Mod': '#9C27B0',
    'PDE':       '#F44336',
    'ATDE':      '#009688',
    'MASTE':     '#E91E63',
    'ST-MW':     '#8BC34A',
    'AMSTE':     '#FF5722',
    'SFBE':      '#3F51B5',
}

ENCODER_TYPE = {
    'Rate':      'Baseline', 'Phase':     'Baseline', 'TTFS':  'Baseline',
    'Delta-Mod': 'Proposed', 'PDE':       'Proposed', 'ATDE':  'Proposed',
    'MASTE':     'Proposed', 'ST-MW':     'Proposed', 'AMSTE': 'Proposed',
    'SFBE':      'Proposed',
}

# ── Architecture ──────────────────────────────────────────────────────────────
# n_traces from your DAS data:
#   June_24: check first file → typically 366 or similar
#   Set manually here if known:
NUM_INPUTS_JUNE  = 366       # n_traces in June_24 data
NUM_HIDDEN1      = 128
NUM_HIDDEN2      = 64
NUM_OUTPUTS      = 2

# ── Temporal binning ──────────────────────────────────────────────────────────
# Each .npy is (n_traces, n_samples). After BIN_FACTOR=4 binning:
#   n_samples=2000 → NUM_STEPS=500 (2000/4)
BIN_FACTOR  = 4
NUM_STEPS   = 500

# ── LIF neuron ────────────────────────────────────────────────────────────────
BETA1       = 0.85   # layer 1 — fast (detects sharp P-wave transients)
BETA2       = 0.90   # layer 2 — medium
BETA3       = 0.95   # output  — slow (accumulates evidence over 1 second)
THRESHOLD   = 1.0
SLOPE       = 25     # fast sigmoid surrogate gradient steepness

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE          = 256
NUM_EPOCHS          = 100
LEARNING_RATE       = 2e-3
NUM_FOLDS           = 5
SEEDS               = [42, 123, 456]
LR_FACTOR           = 0.5
LR_PATIENCE         = 5
EARLY_STOP_PATIENCE = 15
BURNIN_EPOCHS       = 20  # min epochs before early-stop is allowed
CORRECT_RATE        = 0.8
INCORRECT_RATE      = 0.2

# ── Dataset split (Mode 1 within-dataset) ─────────────────────────────────────
TEST_SIZE   = 0.15
SPLIT_SEED  = 42

# ── Energy ────────────────────────────────────────────────────────────────────
ENERGY_PER_SYNOP_PJ = 23.6   # Loihi-2 estimate (pJ per synaptic operation)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Encoders: {ENCODER_NAMES}")
print(f"June_24: {ENCODED_JUNE}")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Dataset Utilities                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def load_labels(encoded_root, dataset_tag=None):
    """
    Load labels.csv for one dataset root.
    Returns (file_ids, labels, source_tags).
    source_tags: array of dataset_tag strings (used for Mode 3 stratification).
    """
    csv_path = os.path.join(encoded_root, 'labels.csv')
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"labels.csv not found: {csv_path}")
    df       = pd.read_csv(csv_path)
    file_ids = df['file_id'].values
    labels   = df['label'].values.astype(int)
    tags     = np.array([dataset_tag or os.path.basename(encoded_root)] * len(labels))
    print(f"  [{dataset_tag or 'data'}] {len(file_ids)} files: "
          f"{np.sum(labels==1)} event + {np.sum(labels==0)} noise")
    return file_ids, labels, tags


def infer_num_inputs(encoded_root, encoder_name):
    """
    Read one .npy file to infer n_traces (NUM_INPUTS).
    Read one .npy file to infer n_traces (NUM_INPUTS).
    """
    enc_dir = os.path.join(encoded_root, encoder_name)
    for f in os.listdir(enc_dir):
        if f.endswith('.npy'):
            arr = np.load(os.path.join(enc_dir, f))
            return arr.shape[0]   # n_traces
    raise FileNotFoundError(f"No .npy files in {enc_dir}")


def load_spike_array(encoded_root, encoder_name, file_id,
                      bin_factor=BIN_FACTOR):
    """
    Load + bin one .npy file.
    Returns float32 array (NUM_STEPS, n_traces).
    """
    path    = os.path.join(encoded_root, encoder_name, f'{file_id}.npy')
    spikes  = np.load(path).astype(np.float32)
    n_tr, n_smp = spikes.shape
    n_bins  = n_smp // bin_factor
    trimmed = spikes[:, :n_bins * bin_factor]
    binned  = trimmed.reshape(n_tr, n_bins, bin_factor).sum(axis=2)
    binned  = (binned > 0).astype(np.float32)
    return binned.T                              # (NUM_STEPS, n_traces)


class DASSpikeDataset(Dataset):
    """
    Lazy-loading dataset for one encoder from one dataset root.
    Each item: (spike_tensor (NUM_STEPS, n_inputs), label int)
    """
    def __init__(self, file_ids, labels, encoded_root, encoder_name,
                  bin_factor=BIN_FACTOR):
        self.file_ids    = np.array(file_ids)
        self.labels      = np.array(labels)
        self.enc_dir     = os.path.join(encoded_root, encoder_name)
        self.bin_factor  = bin_factor

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        path   = os.path.join(self.enc_dir, f'{self.file_ids[idx]}.npy')
        spikes = np.load(path).astype(np.float32)
        n_tr, n_smp = spikes.shape
        n_bins  = n_smp // self.bin_factor
        trimmed = spikes[:, :n_bins * self.bin_factor]
        binned  = trimmed.reshape(n_tr, n_bins, self.bin_factor).sum(axis=2)
        binned  = (binned > 0).astype(np.float32).T   # (NUM_STEPS, n_traces)
        return (torch.tensor(binned, dtype=torch.float32),
                torch.tensor(int(self.labels[idx]), dtype=torch.long))



def available_encoders(encoded_root):
    """Return encoders that have .npy files in encoded_root."""
    avail = []
    for enc in ENCODER_NAMES:
        d = os.path.join(encoded_root, enc)
        if os.path.exists(d) and any(f.endswith('.npy') for f in os.listdir(d)):
            avail.append(enc)
    return avail


# ── Quick dataset check ───────────────────────────────────────────────────────
print("\nChecking available encoders...")
print(f"  June_24: {available_encoders(ENCODED_JUNE)}")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — SNN Model Architecture                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class DAS_SNN(nn.Module):
    """
    Feedforward SNN: Dropout(0.1) → FC(n→128) → LIF → FC(128→64) → LIF → FC(64→2) → LIF

    β hierarchy (all learnable via surrogate gradient):
        β1=0.85 — layer 1: fast decay, detects sharp P-wave transients
        β2=0.90 — layer 2: medium integration
        β3=0.95 — output:  slow accumulation over full 1-second window

    Input dropout (p=0.1):
        Applied per timestep before fc1. Prevents co-adaptation during the
        initial training plateau common with sparse encoders (Rate, Phase, TTFS).
        Disabled automatically during eval (model.eval()).

    Weight init: Kaiming uniform for all FC layers (explicit, not default).

    SynOps formula for FC(n→128→64→2):
        SynOps = (input_spikes × 128) + (L1_spikes × 64) + (L2_spikes × 2)
        Note: dropout is not applied during eval, so SynOps is unaffected.
    """
    def __init__(self, num_inputs=NUM_INPUTS_JUNE, num_hidden1=NUM_HIDDEN1,
                  num_hidden2=NUM_HIDDEN2, num_outputs=NUM_OUTPUTS,
                  num_steps=NUM_STEPS):
        super().__init__()
        self.num_steps  = num_steps
        self.num_inputs = num_inputs
        spike_grad = surrogate.fast_sigmoid(slope=SLOPE)

        # Input dropout — reduces co-adaptation during the initial plateau
        # and stabilises training for sparse encoders (Rate, Phase, TTFS).
        # Applied to the input spike tensor before fc1 only during training.
        self.drop_in = nn.Dropout(p=0.1)

        self.fc1  = nn.Linear(num_inputs, num_hidden1)
        self.lif1 = snn.Leaky(beta=BETA1, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')
        self.fc2  = nn.Linear(num_hidden1, num_hidden2)
        self.lif2 = snn.Leaky(beta=BETA2, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')
        self.fc3  = nn.Linear(num_hidden2, num_outputs)
        self.lif3 = snn.Leaky(beta=BETA3, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')

        # Explicit Kaiming initialisation for all linear layers
        for m in [self.fc1, self.fc2, self.fc3]:
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: (num_steps, batch, num_inputs)
        Returns spk_rec (num_steps, batch, num_outputs),
                mem_rec (num_steps, batch, num_outputs)
        """
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        spk_rec, mem_rec = [], []

        for step in range(self.num_steps):
            inp  = self.drop_in(x[step])          # dropout on input spikes
            cur1 = self.fc1(inp);  spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1); spk2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(spk2); spk3, mem3 = self.lif3(cur3, mem3)
            spk_rec.append(spk3)
            mem_rec.append(mem3)

        return torch.stack(spk_rec), torch.stack(mem_rec)

    def forward_with_internals(self, x):
        """Full forward pass returning all intermediate spike + membrane tensors."""
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        r = {k: [] for k in ['spk1','mem1','spk2','mem2','spk3','mem3']}

        for step in range(self.num_steps):
            inp  = self.drop_in(x[step])
            cur1 = self.fc1(inp);  s1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(s1);   s2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(s2);   s3, mem3 = self.lif3(cur3, mem3)
            r['spk1'].append(s1); r['mem1'].append(mem1)
            r['spk2'].append(s2); r['mem2'].append(mem2)
            r['spk3'].append(s3); r['mem3'].append(mem3)

        return {k: torch.stack(v) for k, v in r.items()}


def load_checkpoint(encoder_name, mode, fold=0, seed=42):
    """
    Load saved model checkpoint.
    mode: 'within' | 'quicktest'
    """
    path = os.path.join(CKPT_DIR, f'{encoder_name}_{mode}_fold{fold}_seed{seed}.pt')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Checkpoint not found: {path}\n"
            f"Run the corresponding training cell first.")
    # Infer num_inputs from checkpoint
    state = torch.load(path, map_location='cpu')
    n_inp = state['fc1.weight'].shape[1]
    model = DAS_SNN(num_inputs=n_inp).to(DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


def compute_synops(model, x_input):
    """Compute SynOps + energy for one sample."""
    model.eval()
    with torch.no_grad():
        internals = model.forward_with_internals(x_input.to(DEVICE))
    n_h1 = model.fc1.out_features
    n_h2 = model.fc2.out_features
    n_out = model.fc3.out_features
    i_sp = x_input.sum().item()
    l1   = internals['spk1'].sum().item()
    l2   = internals['spk2'].sum().item()
    ops  = i_sp*n_h1 + l1*n_h2 + l2*n_out
    return {'synops': ops, 'msynops': ops/1e6,
            'energy_uj': ops * ENERGY_PER_SYNOP_PJ * 1e-6}


# ╔══════════════════════════════════════════════════════════════════════════╗

# ║  CELL 4 — Training + Evaluation Functions                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _metrics(targets, preds, probs):
    """
    Full metric suite for binary event/noise classification.

    Classification metrics:
        accuracy, balanced_acc, precision, recall, f1,
        auc (ROC-AUC), auprc (Average Precision / PR-AUC),
        specificity, fpr, fnr, mcc

    Efficiency metrics are computed separately via compute_efficiency_metrics().
    """
    cm             = confusion_matrix(targets, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        # ── primary ───────────────────────────────────────────────
        'f1':           f1_score(targets, preds, zero_division=0),
        'recall':       recall_score(targets, preds, zero_division=0),
        'fnr':          fn/(fn+tp)   if (fn+tp)>0   else 0.0,
        'fpr':          fp/(fp+tn)   if (fp+tn)>0   else 0.0,
        'auprc':        average_precision_score(targets, probs),
        # ── secondary ─────────────────────────────────────────────
        'accuracy':     accuracy_score(targets, preds),
        'balanced_acc': balanced_accuracy_score(targets, preds),
        'precision':    precision_score(targets, preds, zero_division=0),
        'specificity':  tn/(tn+fp)   if (tn+fp)>0   else 0.0,
        'auc':          roc_auc_score(targets, probs),
        'mcc':          matthews_corrcoef(targets, preds),
        # ── diagnostics ───────────────────────────────────────────
        'cm':           cm.tolist(),
    }


def _forward_batch(model, loader, loss_fn, train=False, optimizer=None):
    """One pass over loader. Returns metrics dict."""
    if train:
        model.train()
    else:
        model.eval()

    total_loss, n = 0.0, 0
    preds_all, tgt_all, prob_all = [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for data, targets in loader:
            data    = data.permute(1, 0, 2).to(DEVICE)   # (T, B, n_inp)
            targets = targets.to(DEVICE)

            spk_rec, _ = model(data)
            loss = loss_fn(spk_rec, targets)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item(); n += 1
            counts = spk_rec.sum(dim=0)
            probs  = torch.softmax(counts.float(), dim=1)
            preds_all.extend(counts.argmax(dim=1).detach().cpu().numpy())
            tgt_all.extend(targets.detach().cpu().numpy())
            prob_all.extend(probs[:, 1].detach().cpu().numpy())

    m = _metrics(np.array(tgt_all), np.array(preds_all), np.array(prob_all))
    m['loss'] = total_loss / max(n, 1)
    return m


def train_one_run(encoder_name, file_ids, labels, train_idx, val_idx,
                   seed, fold, mode, num_inputs=NUM_INPUTS_JUNE,
                   encoded_root=ENCODED_JUNE):
    """
    Train one fold/seed combination.
    mode: string tag for checkpoint naming ('within'|'quicktest')
    Returns best validation metrics dict + saves checkpoint.
    """
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()   # prevent VRAM fragmentation across 15 runs

    train_ds = DASSpikeDataset(file_ids[train_idx], labels[train_idx],
                                encoded_root, encoder_name)
    val_ds   = DASSpikeDataset(file_ids[val_idx], labels[val_idx],
                                encoded_root, encoder_name)
    tr_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=False)   # Jetson: no fork
    va_ld = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

    model     = DAS_SNN(num_inputs=num_inputs).to(DEVICE)
    loss_fn   = SF.mse_count_loss(correct_rate=CORRECT_RATE,
                                   incorrect_rate=INCORRECT_RATE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_f1, best_meta, best_w = -1.0, None, None
    best_epoch, no_imp = 0, 0

    for epoch in range(NUM_EPOCHS):
        t0  = time.time()
        trn = _forward_batch(model, tr_ld, loss_fn, train=True, optimizer=optimizer)
        val = _forward_batch(model, va_ld, loss_fn)
        scheduler.step(val['loss'])
        lr = optimizer.param_groups[0]['lr']

        # Update best checkpoint — ignore degenerate all-positive solutions
        # (F1≈0.667 with FNR≈0 on balanced data = predicting all events).
        # Only accept improvements after the burn-in period.
        is_degenerate = (val['f1'] <= 0.70 and val['fnr'] < 0.05)
        past_burnin   = (epoch + 1) >= BURNIN_EPOCHS

        if val['f1'] > best_f1 and not is_degenerate:
            best_f1 = val['f1']; best_meta = val.copy()
            best_w  = copy.deepcopy(model.state_dict())
            best_epoch = epoch+1; no_imp = 0
        elif past_burnin:
            no_imp += 1

        deg_flag = ' [DEG]' if is_degenerate else ''
        burn_flag = ' [BURN]' if not past_burnin else ''
        print(f"  Ep {epoch+1:>3d}  "
              f"| TR  loss={trn['loss']:.4f} acc={trn['accuracy']:.3f} "
              f"f1={trn['f1']:.3f} auc={trn['auc']:.3f}  "
              f"| VA  loss={val['loss']:.4f} acc={val['accuracy']:.3f} "
              f"f1={val['f1']:.3f} auc={val['auc']:.3f}  "
              f"lr={lr:.2e}  {'★' if no_imp==0 and past_burnin else ' '}"
              f"{deg_flag}{burn_flag}  ({time.time()-t0:.1f}s)")

        # Early stop only after burn-in
        if past_burnin and no_imp >= EARLY_STOP_PATIENCE:
            print(f"   Early stop @epoch {epoch+1}  (best F1={best_f1:.4f} @ep{best_epoch})")
            break

    if best_w is None:
        # All epochs were degenerate — use final state and warn
        print(f"   WARNING: no non-degenerate checkpoint found for "
              f"{encoder_name} fold={fold} seed={seed}. Using final weights.")
        best_w    = copy.deepcopy(model.state_dict())
        best_meta = val.copy()
        best_epoch = NUM_EPOCHS

    model.load_state_dict(best_w)
    best_meta.update({
        'encoder': encoder_name, 'fold': fold, 'seed': seed,
        'mode': mode, 'best_epoch': best_epoch,
        'betas': {'b1': model.lif1.beta.item(),
                  'b2': model.lif2.beta.item(),
                  'b3': model.lif3.beta.item()},
    })
    ckpt = os.path.join(CKPT_DIR, f'{encoder_name}_{mode}_fold{fold}_seed{seed}.pt')
    torch.save(best_w, ckpt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()   # free VRAM before next run
    return best_meta


def evaluate_loader(model, loader, loss_fn=None):
    """Evaluate model on a DataLoader, return metrics dict + probs."""
    if loss_fn is None:
        loss_fn = SF.mse_count_loss(correct_rate=CORRECT_RATE,
                                     incorrect_rate=INCORRECT_RATE)
    return _forward_batch(model, loader, loss_fn)



def compute_efficiency_metrics(model, loader, num_inputs, num_steps=NUM_STEPS):
    """
    Compute spike-rate and SynOps efficiency metrics over a full DataLoader.

    Returns dict with:
        input_spike_rate  — mean fraction of input neurons firing per timestep
                            (spikes / (T * n_inputs) per sample)
        l1_spike_rate     — same for hidden layer 1 (128 units)
        l2_spike_rate     — same for hidden layer 2 (64 units)
        synops            — mean total synaptic operations per sample
        msynops           — synops / 1e6
        energy_uj         — mean energy estimate (µJ) using Loihi-2 figure

    SynOps formula:
        (input_spikes × 128) + (L1_spikes × 64) + (L2_spikes × 2)

    Note: model.eval() is enforced so Dropout is disabled — SynOps reflects
    true inference cost, not the dropout-augmented training cost.
    """
    model.eval()
    n_h1  = model.fc1.out_features   # 128
    n_h2  = model.fc2.out_features   # 64
    n_out = model.fc3.out_features   # 2

    inp_rates, l1_rates, l2_rates, synops_all = [], [], [], []
    t_total = 0.0

    with torch.no_grad():
        for data, _ in loader:
            data = data.permute(1, 0, 2).to(DEVICE)  # (T, B, n_inp)
            B    = data.shape[1]
            t0   = time.time()
            internals = model.forward_with_internals(data)
            t_total  += time.time() - t0

            # spike counts per sample
            inp_spikes = data.sum(dim=0).sum(dim=1)          # (B,)
            l1_spikes  = internals['spk1'].sum(dim=0).sum(dim=1)
            l2_spikes  = internals['spk2'].sum(dim=0).sum(dim=1)

            # normalise to rate (fraction of neurons × timesteps)
            inp_rates.extend((inp_spikes / (num_steps * num_inputs)).cpu().tolist())
            l1_rates.extend( (l1_spikes  / (num_steps * n_h1)).cpu().tolist())
            l2_rates.extend( (l2_spikes  / (num_steps * n_h2)).cpu().tolist())

            # SynOps per sample
            s = (inp_spikes * n_h1 + l1_spikes * n_h2 + l2_spikes * n_out)
            synops_all.extend(s.cpu().tolist())

    n_samples    = len(inp_rates)
    mean_synops  = float(np.mean(synops_all))
    mean_energy  = mean_synops * ENERGY_PER_SYNOP_PJ * 1e-6   # pJ → µJ
    inf_time_ms  = (t_total / n_samples) * 1e3                 # s → ms

    return {
        'input_spike_rate': float(np.mean(inp_rates)),
        'l1_spike_rate':    float(np.mean(l1_rates)),
        'l2_spike_rate':    float(np.mean(l2_rates)),
        'synops':           mean_synops,
        'msynops':          mean_synops / 1e6,
        'energy_uj':        mean_energy,
        'inference_time_ms': inf_time_ms,
    }

def save_results(rows, filename):
    """Append rows (list of dicts) to a CSV in RESULTS_DIR."""
    path   = os.path.join(RESULTS_DIR, filename)
    new_df = pd.DataFrame(rows)
    if os.path.exists(path):
        new_df = pd.concat([pd.read_csv(path), new_df], ignore_index=True)
    new_df.to_csv(path, index=False)
    print(f"  Results → {path}")


# ╔══════════════════════════════════════════════════════════════════════════╗

# ║  CELL 5 — MODE 1: Within-dataset 5-fold CV on June_24                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
"""
WITHIN-DATASET EVALUATION
───────────────────────────────────────────────────────────────────────────
Uses ONLY June_24 (2,000 event + 2,000 noise = 4,000 files).

Design:
  1. Fixed 15% held-out test set (600 files) — same for all encoders.
  2. Remaining 85% (3,400 files) → 5-fold stratified CV.
  3. 3 seeds per fold → 15 runs per encoder → mean±std reported.
  4. Best checkpoint → evaluate once on held-out test set.

Run one encoder:   mode1_single('Rate')
Run all encoders:  mode1_all()
Evaluate test set: mode1_test_eval('Rate')
"""

def mode1_get_split(encoded_root=ENCODED_JUNE):
    """Fixed train/val pool + test split. Returns arrays."""
    file_ids, labels, _ = load_labels(encoded_root, 'june')
    tv_ids, te_ids, tv_lab, te_lab = train_test_split(
        file_ids, labels, test_size=TEST_SIZE,
        stratify=labels, random_state=SPLIT_SEED)
    print(f"\nMode 1 split (seed={SPLIT_SEED}):")
    print(f"  Train/Val: {len(tv_ids):>5d}  "
          f"({np.sum(tv_lab==1)} ev + {np.sum(tv_lab==0)} no)")
    print(f"  Test:      {len(te_ids):>5d}  "
          f"({np.sum(te_lab==1)} ev + {np.sum(te_lab==0)} no)")
    return tv_ids, tv_lab, te_ids, te_lab


def mode1_single(encoder_name, n_folds=NUM_FOLDS, seeds=SEEDS,
                  encoded_root=ENCODED_JUNE):
    """
    Train encoder on June_24 with stratified CV.
    Returns list of per-fold/seed result dicts.
    """
    n_inp = infer_num_inputs(encoded_root, encoder_name)
    tv_ids, tv_lab, _, _ = mode1_get_split(encoded_root)

    print(f"\n{'='*65}")
    print(f"MODE 1 — Within-dataset CV: {encoder_name}")
    print(f"  Dataset: June_24 | n_inputs={n_inp} | "
          f"{n_folds} folds × {len(seeds)} seeds")
    print(f"{'='*65}")

    skf     = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SPLIT_SEED)
    results = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(tv_ids, tv_lab)):
        for seed in seeds:
            print(f"\n  ── Fold {fold+1}/{n_folds}  Seed {seed} ──")
            r = train_one_run(
                encoder_name, tv_ids, tv_lab, tr_idx, va_idx,
                seed=seed, fold=fold, mode='within',
                num_inputs=n_inp, encoded_root=encoded_root)
            results.append(r)
            print(f"     Best val F1={r['f1']:.3f}  "
                  f"AUC={r['auc']:.3f}  FNR={r['fnr']:.3f}")

    # Print fold summary
    df = pd.DataFrame(results)
    print(f"\n  WITHIN-CV SUMMARY ({encoder_name}):")
    primary   = ['f1', 'recall', 'fnr', 'fpr', 'auprc']
    secondary = ['accuracy', 'balanced_acc', 'precision', 'specificity', 'auc', 'mcc']
    print(f"  {'─'*50}")
    print(f"  Primary metrics:")
    for metric in primary:
        if metric in df.columns:
            print(f"    {metric:14s}: {df[metric].mean():.4f} ± {df[metric].std():.4f}")
    print(f"  Secondary metrics:")
    for metric in secondary:
        if metric in df.columns:
            print(f"    {metric:14s}: {df[metric].mean():.4f} ± {df[metric].std():.4f}")

    save_results(results, 'mode1_within_cv_results.csv')
    return results


def mode1_test_eval(encoder_name, encoded_root=ENCODED_JUNE):
    """
    Evaluate best Mode 1 checkpoint on the held-out June_24 test set.
    'Best' = fold/seed with highest val F1 from mode1_within_cv_results.csv.
    """
    csv_path = os.path.join(RESULTS_DIR, 'mode1_within_cv_results.csv')
    if not os.path.exists(csv_path):
        raise FileNotFoundError("Run mode1_single() first.")
    df  = pd.read_csv(csv_path)
    sub = df[df['encoder'] == encoder_name]
    if len(sub) == 0:
        raise ValueError(f"No Mode 1 results for {encoder_name}")
    best = sub.loc[sub['f1'].idxmax()]
    fold, seed = int(best['fold']), int(best['seed'])
    print(f"\nMode 1 test eval — {encoder_name}  "
          f"(best val F1={best['f1']:.3f}, fold={fold}, seed={seed})")

    _, _, te_ids, te_lab = mode1_get_split(encoded_root)
    n_inp = infer_num_inputs(encoded_root, encoder_name)
    model = load_checkpoint(encoder_name, 'within', fold=fold, seed=seed)
    te_ds = DASSpikeDataset(te_ids, te_lab, encoded_root, encoder_name)
    te_ld = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    m = evaluate_loader(model, te_ld)

    # Efficiency metrics on test set (model.eval() enforced inside)
    print(f"  Computing efficiency metrics on test set...")
    eff = compute_efficiency_metrics(model, te_ld,
                                     num_inputs=n_inp, num_steps=NUM_STEPS)
    m.update(eff)
    m.update({'encoder': encoder_name, 'mode': 'mode1_test',
               'fold': fold, 'seed': seed,
               'n_test': len(te_ids)})
    _print_test_metrics(m, f"Mode 1 Test Set — {encoder_name} (June_24)")
    save_results([m], 'mode1_test_results.csv')
    return m


def mode1_all(n_folds=NUM_FOLDS, seeds=SEEDS, encoded_root=ENCODED_JUNE,
              run_test_eval=True):
    """
    Run Mode 1 for all available encoders in June_24.
    If run_test_eval=True, automatically calls mode1_test_eval() for each
    encoder after CV training — this computes efficiency metrics on the
    held-out test set and saves to mode1_test_results.csv.
    """
    avail = available_encoders(encoded_root)
    print(f"\nMode 1 — Running {len(avail)} encoders on June_24")
    all_results = []
    for enc in avail:
        try:
            r = mode1_single(enc, n_folds=n_folds, seeds=seeds,
                              encoded_root=encoded_root)
            all_results.extend(r)
            if run_test_eval:
                try:
                    mode1_test_eval(enc, encoded_root=encoded_root)
                except Exception as ex:
                    print(f"    test_eval {enc}: {ex}")
        except Exception as ex:
            print(f"    {enc}: {ex}")
    return all_results


# ╔══════════════════════════════════════════════════════════════════════════╗

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Summary + Comparison Plots                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _print_test_metrics(m, title):
    """Pretty-print full classification + efficiency metrics."""
    cm = np.array(m['cm'])
    tn, fp, fn, tp = cm.ravel()
    w = 60
    print(f"\n{'─'*w}")
    print(f"  {title}")
    print(f"{'─'*w}")
    print(f"  ── Classification (primary) ─────────────────────────")
    print(f"  F1 Score      : {m['f1']:.4f}")
    print(f"  Recall        : {m['recall']:.4f}  ({tp} detected / {fn+tp} events)")
    print(f"  FNR           : {m['fnr']:.4f}  ({fn} missed / {fn+tp} events)")
    print(f"  FPR           : {m['fpr']:.4f}  ({fp} false alarms / {tn+fp} noise)")
    print(f"  AUPRC         : {m.get('auprc', float('nan')):.4f}")
    print(f"  ── Classification (secondary) ───────────────────────")
    print(f"  Accuracy      : {m['accuracy']:.4f}")
    print(f"  Balanced Acc  : {m.get('balanced_acc', float('nan')):.4f}")
    print(f"  Precision     : {m['precision']:.4f}")
    print(f"  Specificity   : {m.get('specificity', float('nan')):.4f}")
    print(f"  AUC-ROC       : {m['auc']:.4f}")
    print(f"  MCC           : {m.get('mcc', float('nan')):.4f}")
    if 'input_spike_rate' in m:
        print(f"  ── Efficiency ───────────────────────────────────────")
        print(f"  Input spike rate : {m['input_spike_rate']:.4f}  "
              f"(L1: {m['l1_spike_rate']:.4f}  L2: {m['l2_spike_rate']:.4f})")
        print(f"  SynOps           : {m['msynops']:.4f} MSynOps  "
              f"({m['synops']:.0f} total)")
        print(f"  Energy estimate  : {m['energy_uj']:.4f} µJ  "
              f"(Loihi-2 @ {ENERGY_PER_SYNOP_PJ} pJ/SynOp)")
        print(f"  Inference time   : {m.get('inference_time_ms', 0):.3f} ms/sample")
    print(f"  ── Confusion Matrix ─────────────────────────────────")
    print(f"              Pred Noise  Pred Event")
    print(f"  True Noise    {tn:>6d}      {fp:>6d}")
    print(f"  True Event    {fn:>6d}      {tp:>6d}")


def print_final_summary():
    """
    Print two comparison tables:
      Table 1 — Classification metrics (from mode1_within_cv_results.csv)
      Table 2 — Efficiency metrics     (from mode1_test_results.csv)
    """
    cv_path   = os.path.join(RESULTS_DIR, 'mode1_within_cv_results.csv')
    test_path = os.path.join(RESULTS_DIR, 'mode1_test_results.csv')

    # ── Table 1: Classification ────────────────────────────────────────────
    if os.path.exists(cv_path):
        df = pd.read_csv(cv_path)
        best_f1 = df.groupby('encoder')['f1'].mean().max()
        print(f"\n{'='*100}")
        print(f"  TABLE 1 — Classification Metrics  (Mode 1 Within-CV, mean±std over 15 runs)")
        print(f"{'='*100}")
        hdr = (f"{'Encoder':>12}  {'Type':>9}  {'F1':>12}  {'Recall':>12}  "
               f"{'FNR':>12}  {'FPR':>12}  {'AUPRC':>12}  {'AUC':>12}  {'MCC':>8}")
        print(hdr)
        print('─' * len(hdr))
        for enc in ENCODER_NAMES:
            sub = df[df['encoder']==enc]
            if len(sub)==0: continue
            etype = ENCODER_TYPE.get(enc,'')
            flag  = ' ◀' if sub['f1'].mean() == best_f1 else ''
            def ms(col):  # mean±std helper
                return (f"{sub[col].mean():.4f}±{sub[col].std():.3f}"
                        if col in sub.columns else '  N/A      ')
            print(f"  {enc:>12}  {etype:>9}  {ms('f1'):>12}  {ms('recall'):>12}  "
                  f"{ms('fnr'):>12}  {ms('fpr'):>12}  {ms('auprc'):>12}  "
                  f"{ms('auc'):>12}  "
                  f"{sub['mcc'].mean():>7.4f}{flag}" if 'mcc' in sub.columns
                  else f"  {enc:>12}  {etype:>9}  {ms('f1'):>12}{flag}")
    else:
        print("No CV results — run mode1_all() first.")

    # ── Table 2: Efficiency ────────────────────────────────────────────────
    if os.path.exists(test_path):
        df2 = pd.read_csv(test_path)
        print(f"\n{'='*90}")
        print(f"  TABLE 2 — Efficiency Metrics  (Mode 1 Test Set, best checkpoint per encoder)")
        print(f"{'='*90}")
        hdr2 = (f"{'Encoder':>12}  {'Type':>9}  "
                f"{'InSpikeRate':>12}  {'L1SpikeRate':>12}  "
                f"{'MSynOps':>10}  {'Energy(µJ)':>12}  {'InfTime(ms)':>12}")
        print(hdr2)
        print('─' * len(hdr2))
        for enc in ENCODER_NAMES:
            sub = df2[df2['encoder']==enc]
            if len(sub)==0: continue
            etype = ENCODER_TYPE.get(enc,'')
            def g(col):
                return f"{sub[col].mean():.4f}" if col in sub.columns else '  N/A'
            print(f"  {enc:>12}  {etype:>9}  "
                  f"{g('input_spike_rate'):>12}  {g('l1_spike_rate'):>12}  "
                  f"{g('msynops'):>10}  {g('energy_uj'):>12}  "
                  f"{g('inference_time_ms'):>12}")
    else:
        print("\nNo test results — run mode1_test_eval() or mode1_all(run_test_eval=True).")


def plot_mode1_results(metric='f1'):
    """
    Bar chart of Mode 1 within-CV results per encoder with error bars.
    Baselines shaded in gray; proposed encoders coloured.
    """
    p = os.path.join(RESULTS_DIR, 'mode1_within_cv_results.csv')
    if not os.path.exists(p):
        print("No results found. Run mode1_all() first.")
        return
    df    = pd.read_csv(p)
    stats = df.groupby('encoder')[metric].agg(['mean','std'])
    encs  = [e for e in ENCODER_NAMES if e in stats.index]
    means = [stats.loc[e, 'mean'] for e in encs]
    stds  = [stats.loc[e, 'std']  for e in encs]
    colors = [ENCODER_COLORS.get(e,'#888') for e in encs]

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(range(len(encs)), means, yerr=stds, capsize=4,
                   color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+s+0.01,
                f'{m:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(encs)))
    ax.set_xticklabels(encs, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel(metric.upper(), fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.grid(axis='y', alpha=0.3)
    # Shade baselines
    for i, enc in enumerate(encs):
        if ENCODER_TYPE.get(enc) == 'Baseline':
            ax.axvspan(i-0.45, i+0.45, alpha=0.08, color='gray', zorder=0)
    ax.set_title(f'Mode 1 Within-CV — {metric.upper()} per Encoder\n'
                 f'(Gray shading = baselines | 5-fold CV × 3 seeds | June_24)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, f'mode1_{metric}_comparison.png')
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")

def plot_efficiency_frontier():
    """
    Scatter plot: F1 (y) vs MSynOps (x) per encoder.
    Ideal encoders sit in the top-left (high F1, low compute).
    Loaded from mode1_test_results.csv.
    """
    p = os.path.join(RESULTS_DIR, 'mode1_test_results.csv')
    if not os.path.exists(p):
        print("No test results — run mode1_test_eval() first.")
        return
    df = pd.read_csv(p)
    if 'msynops' not in df.columns:
        print("No efficiency metrics in test results — run mode1_test_eval().")
        return

    fig, ax = plt.subplots(figsize=(10, 6))
    for enc in ENCODER_NAMES:
        sub = df[df['encoder']==enc]
        if len(sub)==0: continue
        x = sub['msynops'].mean()
        y = sub['f1'].mean()
        c = ENCODER_COLORS.get(enc, '#888')
        marker = 'D' if ENCODER_TYPE.get(enc) == 'Baseline' else 'o'
        ax.scatter(x, y, color=c, s=120, marker=marker,
                   zorder=3, edgecolors='white', linewidth=1.0)
        ax.annotate(enc, (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=8, color=c)

    ax.set_xlabel('MSynOps (× 10⁶ synaptic operations per inference)', fontsize=11)
    ax.set_ylabel('F1 Score (test set)', fontsize=11)
    ax.set_title('Efficiency Frontier — F1 vs Computational Cost\n'
                 '(◆ = Baseline  ● = Proposed  |  top-left = best trade-off)',
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim(0.5, 1.05)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'efficiency_frontier.png')
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")


# ╔══════════════════════════════════════════════════════════════════════════╗

# ║  CELL 9 — Quick Test + Usage Examples                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def quick_test(encoder_name='Rate', n_files=50, encoded_root=ENCODED_JUNE):
    """
    Sanity check: train 1 fold × 1 seed on max n_files, print F1.
    Use before committing to a full run (~2 min).
    """
    n_inp = infer_num_inputs(encoded_root, encoder_name)
    file_ids, labels, _ = load_labels(encoded_root, 'test')

    rng  = np.random.default_rng(42)
    sel  = rng.choice(len(file_ids), min(n_files, len(file_ids)), replace=False)
    ids  = file_ids[sel]; lab = labels[sel]
    n_tr = int(0.8 * len(ids))
    tr_idx = np.arange(n_tr); va_idx = np.arange(n_tr, len(ids))

    global NUM_EPOCHS, EARLY_STOP_PATIENCE
    orig_ep = NUM_EPOCHS
    orig_es = EARLY_STOP_PATIENCE

    print(f"\nQuick test: {encoder_name} | {n_files} files | "
          f"3 epochs | n_inputs={n_inp}")

    NUM_EPOCHS          = 3
    EARLY_STOP_PATIENCE = 10

    r = train_one_run(encoder_name, ids, lab, tr_idx, va_idx,
                       seed=42, fold=0, mode='quicktest',
                       num_inputs=n_inp, encoded_root=encoded_root)

    NUM_EPOCHS          = orig_ep
    EARLY_STOP_PATIENCE = orig_es
    print(f"\n  Quick test result: F1={r['f1']:.3f}  AUC={r['auc']:.3f}")
    print(f"  Encoder loads + trains correctly.")
    return r


"""
═══════════════════════════════════════════════════════════════════════════
USAGE GUIDE — run each cell in order
═══════════════════════════════════════════════════════════════════════════

── 1. Sanity check (always run first, ~2 min) ───────────────────────────
quick_test('AMSTE', n_files=100)

── 2. MODE 1: Within-dataset CV on June_24 ──────────────────────────────
# One encoder at a time (recommended — monitor first):
results = mode1_single('Rate')
results = mode1_single('Delta-Mod')
results = mode1_single('AMSTE')

# All encoders (long — ~8-12 hrs total):
all_results = mode1_all()

# Evaluate best checkpoint on held-out test set:
mode1_test_eval('Rate')
mode1_test_eval('AMSTE')

── 3. Summary + plots (run after mode1_all()) ───────────────────────────
print_final_summary()               # Tables 1 (classification) + 2 (efficiency)
plot_mode1_results(metric='f1')
plot_mode1_results(metric='auc')
plot_mode1_results(metric='fnr')
plot_mode1_results(metric='auprc')
plot_efficiency_frontier()          # F1 vs MSynOps scatter

═══════════════════════════════════════════════════════════════════════════
"""

all_results = mode1_all() 

print_final_summary()
plot_mode1_results(metric='f1')
plot_mode1_results(metric='fnr')
plot_mode1_results(metric='auprc')
plot_efficiency_frontier()